In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
text = "the cat ate the rat the cat sat on the log the dog ate the cat"
words=text.split()
unq_words= {w: i for i,w in enumerate( sorted(set(words)))}  #token id for each uinique word
unq_words["<PAD>"]=len(unq_words)  #token for the padding (padding is to make the total tokens in a sentence equal)
id_to_word={i:w for w ,i in unq_words.items()}
unq_words_size=len(unq_words)
def encode(sentence):
  return [unq_words[w] for w in sentence.split()]
def decode(ids):
  return  " ".join(id_to_word[i] for i in ids)
print(unq_words_size)


9


In [ ]:
d_model=16 # vector dimension to indicate the word
d_ff=64 #hidden-size inside feed forward network
seq_len=6 #length of each sequence take from the input string
lrng_rate=3e-3 #learning rate
epochs=500



In [ ]:
class Transformer(nn.Module):
  def __init__(self):
    super().__init__()                                          #unique_words_size=9,d_model=16
    self.embedding=nn.Embedding(unq_words_size,d_model)     #9x16 matrix where each row indicates each word in its vectorization
    self.W_Q=nn.Linear(d_model,d_model)                      # 16x16 matrix for weights of conversion of each word's dimesion s into queries of ame dimension
    self.W_K=nn.Linear(d_model,d_model)   #keys
    self.W_V=nn.Linear(d_model,d_model)   #values
    self.ff1 = nn.Linear(d_model, d_ff)     # 16->64
    self.ff2 = nn.Linear(d_ff, d_model)     # 64->16  this introduces non linearity due to RElu and therfore will loose unecessary data
    self.output_head=nn.Linear(d_model,unq_words_size)
    self.norm1=nn.LayerNorm(d_model)  # one used before attention
    self.norm2=nn.LayerNorm(d_model)  # another before ffn
    mask=torch.tril(torch.ones(seq_len,seq_len))   #make lower traingular marix of ones o ensure casuality satisfied
    self.register_buffer("mask",mask)
    pe=self._build_pe(seq_len,d_model) # posituonal encoding with soe inbuilt function involving sines and cosines
    self.register_buffer("pe",pe.unsqueeze(0))

  def forward(self,token_ids):
    B,S=token_ids.shape
    x=self.embedding(token_ids)*math.sqrt(d_model)
    x=x+self.pe[:,:S]
    Q=self.W_Q(x)    #(B,S,16)
    K=self.W_K(x)
    V=self.W_V(x)
    scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(d_model)
    scores = scores.masked_fill(self.mask[:S,:S] == 0, float('-inf')) #makes soft max power -inf for all zeroes to make them independent on future
    attn_weights = F.softmax(scores, dim=-1)
    attn_out = torch.bmm(attn_weights, V)
    x = self.norm1(x + attn_out)  #residual addition
    ff = F.relu(self.ff1(x))
    ff = self.ff2(ff)
    x = self.norm2(x + ff)     # residual + layer norm (B, S, 16)
    logits = self.output_head(x)    # (B, S, 11)
    return logits
  @staticmethod
  def _build_pe(seq_len, d):
    pe = torch.zeros(seq_len, d)
    pos = torch.arange(seq_len).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe


In [ ]:
model =Transformer()
# building training data
all_ids = encode(text)
inputs, targets = [], []
for i in range(len(all_ids) - seq_len):
    inputs.append(all_ids[i : i + seq_len])
    targets.append(all_ids[i+1 : i + seq_len + 1])

inputs  = torch.tensor(inputs)
targets = torch.tensor(targets)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lrng_rate)
for epoch in range(epochs):
    logits = model(inputs)
    loss = criterion(logits.view(-1, unq_words_size),targets.view(-1))
    optimizer.zero_grad()   # clear gradients from previous step
    loss.backward()         # compute gradients
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"  Epoch {epoch+1:4d} , loss = {loss.item():.4f}")

print()



  Epoch  100 , loss = 0.1811
  Epoch  200 , loss = 0.1267
  Epoch  300 , loss = 0.1210
  Epoch  400 , loss = 0.1193
  Epoch  500 , loss = 0.1180



In [ ]:
def generate(prompt, n_words=5):
    model.eval()
    ids = encode(prompt)
    with torch.no_grad():
        for _ in range(n_words):
            inp = ids[-seq_len:]
            # pad if shorter than SEQ_LEN
            while len(inp) < seq_len:
                inp = [unq_words["<PAD>"]] + inp
            x = torch.tensor([inp])
            logits = model(x)
            # take logits at the last position to predict what comes next
            next_logits = logits[0, -1, :]
            next_id = next_logits.argmax().item()   # greedy decoding
            ids.append(next_id)
    return decode(ids)



for prompt in ["the cat", "the dog", "sat on the"]:
    print(f"  '{prompt}' -> {generate(prompt)}")

  'the cat' -> the cat ate the cat sat on
  'the dog' -> the dog ate the cat sat on
  'sat on the' -> sat on the log the dog ate the
